In [7]:
!pip install datasets
import pandas as pd
from datasets import load_dataset
raw_dataset = load_dataset("wangrongsheng/ag_news")
df=pd.DataFrame(raw_dataset["train"])
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [11]:
import re
def clean_text(text):
    text = str(text).lower()                  # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text)      # Remove numbers, punctuation, and special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text
    # Apply the function to the text column
df['text'] = df['text'].apply(clean_text)

# Display the cleaned data
df.head()

,text,label
0,wall st bears claw back into the black reuters...,2
1,carlyle looks toward commercial aerospace reut...,2
2,oil and economy cloud stocks outlook reuters r...,2
3,iraq halts oil exports from main southern pipe...,2
4,oil prices soar to alltime record posing new m...,2


In [13]:
# Step 3: Tokenization

from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split

vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")

# Split the data into training and testing sets for texts and labels
train_texts, test_texts, train_labels, test_labels = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

print(train_sequences[0])

[371, 1913, 1, 4830, 1, 163, 3, 1109, 382, 704, 837, 3, 193, 2, 3249, 979, 158, 1, 3331, 1]


In [14]:
# Step 4: Padding

from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 50

train_padded = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

test_padded = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(train_padded.shape)

(96000, 50)


In [15]:
# Step 5: One-Hot Encoding

from tensorflow.keras.utils import to_categorical

num_classes = 4

train_labels = to_categorical(train_labels, num_classes)
test_labels = to_categorical(test_labels, num_classes)

print(train_labels[0])

[1. 0. 0. 0.]


In [16]:
# Step 6: Build RNN

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

embedding_dim = 64

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),

    SimpleRNN(64),

    Dense(num_classes, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Step 7: Compile and Train

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_padded,
    train_labels,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

In [18]:
# Step 8: Evaluate

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

loss, accuracy = model.evaluate(
    test_padded,
    test_labels
)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.2515 - loss: 1.3960
Test Loss : 1.395965337753296
Test Accuracy : 0.2514583468437195
